# Modelo base K-Means

In [1]:
# Importaciones
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# Carpeta actual
directorio_actual = Path.cwd().resolve()
# carpeta raíz
PROJECT_ROOT = directorio_actual.parent

if not (PROJECT_ROOT / "data").is_dir():
    raise FileNotFoundError(
        f"No se encontró la capeta 'data' dentro de {PROJECT_ROOT}"
    )

# Rutas de carpetas donde se ubican los archivos
DATASETS_DIR = PROJECT_ROOT / "data" / "processed" / "discretizaciones"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

# Ruta de archivos
ruta_datasets = DATASETS_DIR / "interpolacion_50.parquet"
df_interpolacion_50 = pd.read_parquet(ruta_datasets)

# Ruta catálogo
ruta_catalogo = INTERIM_DIR / "catalogo_maestro_final.csv"
df_catalogo = pd.read_csv(ruta_catalogo)

df_catalogo.head()


,observacion_id,star_id_base,star_id,proyecto_origen,region_origen,region_catalogo,subtipo_rrlyrae,grupo_analisis,dos_observaciones,periodo_catalogo,...,subtipo_coincidente,error_periodo_catalogo,epoca_catalogo,n_obs_folding,archivo_folding,en_catalogo_final,modo_plegado,periodo_plegado,epoca_plegado,plegado_corregido
0,OGLEIII_OGLE-LMC-RRLYR-00001,OGLE-LMC-RRLYR-00001,OGLE-LMC-RRLYR-00001,OGLE-III,LMC,LMC,RRab,principal,True,0.634752,...,True,0.000002,2184.64705,332,..\data\interim\curvas_plegadas_corregidas.par...,True,modo_fundamental,0.634752,2184.64705,False
1,OGLEIII_OGLE-LMC-RRLYR-00002,OGLE-LMC-RRLYR-00002,OGLE-LMC-RRLYR-00002,OGLE-III,LMC,LMC,RRab,principal,True,0.625525,...,True,0.000003,2171.36808,359,..\data\interim\curvas_plegadas_corregidas.par...,True,modo_fundamental,0.625525,2171.36808,False
2,OGLEIII_OGLE-LMC-RRLYR-00003,OGLE-LMC-RRLYR-00003,OGLE-LMC-RRLYR-00003,OGLE-III,LMC,LMC,RRab,principal,True,0.656497,...,True,0.000001,2171.43774,360,..\data\interim\curvas_plegadas_corregidas.par...,True,modo_fundamental,0.656497,2171.43774,False
3,OGLEIII_OGLE-LMC-RRLYR-00004,OGLE-LMC-RRLYR-00004,OGLE-LMC-RRLYR-00004,OGLE-III,LMC,LMC,RRab,principal,True,0.512374,...,True,0.000001,2171.54298,360,..\data\interim\curvas_plegadas_corregidas.par...,True,modo_fundamental,0.512374,2171.54298,False
4,OGLEIII_OGLE-LMC-RRLYR-00005,OGLE-LMC-RRLYR-00005,OGLE-LMC-RRLYR-00005,OGLE-III,LMC,LMC,RRab,principal,True,0.643352,...,True,0.000003,2171.58055,359,..\data\interim\curvas_plegadas_corregidas.par...,True,modo_fundamental,0.643352,2171.58055,False


In [3]:
# Selección de metadatos

columnas_metadata = [
    "observacion_id",
    "star_id_base",
    "proyecto_origen",
    "region_origen",
    "subtipo_rrlyrae",
    "grupo_analisis"
]

metadata_catalogo = df_catalogo[columnas_metadata].copy()

In [4]:
# Se cruzan metadatos con sus curvas correspondientes
metadata = df_interpolacion_50[["observacion_id"]].merge(
    metadata_catalogo,
    on="observacion_id",
    how="left",
    sort=False,
    validate="one_to_one",
)

# Se seleccionan los grupos de análisis
metadata_principal = metadata.loc[
    metadata["grupo_analisis"] == "principal"
].copy()


estrellas = np.sort(metadata_principal["star_id_base"].unique())

In [5]:
# Preparación de las particiones
SEED_PARTICION = 42

rng_particion = np.random.default_rng(SEED_PARTICION)
estrellas_mezcladas = rng_particion.permutation(estrellas)

n_train = int(len(estrellas_mezcladas) * 0.6)
n_val = int(len(estrellas_mezcladas) * 0.2)

estrellas_train = estrellas_mezcladas[:n_train]
estrellas_val = estrellas_mezcladas[n_train:n_train+n_val]
estrellas_test = estrellas_mezcladas[n_train+n_val:]

metadata_principal["particion"] = ""
mascara_train = metadata_principal["star_id_base"].isin(estrellas_train)
metadata_principal.loc[mascara_train, "particion"] = "train"

mascara_val = metadata_principal["star_id_base"].isin(estrellas_val)
metadata_principal.loc[mascara_val, "particion"] = "val"

mascara_test = metadata_principal["star_id_base"].isin(estrellas_test)
metadata_principal.loc[mascara_test, "particion"] = "test"



In [6]:
if not metadata_principal["particion"].isin(["train", "val", "test"]).all():
    raise ValueError("Hay curvas sin una partición válida.")

In [7]:
particiones_por_estrella = (
    metadata_principal.groupby("star_id_base")["particion"].nunique()
)

if (particiones_por_estrella > 1).any():
    raise ValueError("Hay estrellas presentes en más de una partición.")

In [8]:
# Exportación de las particiones
PARTICIONES_DIR = PROJECT_ROOT / "data" / "processed" / "particiones"
PARTICIONES_DIR.mkdir(parents=True, exist_ok=True)

metadata_principal["seed_particion"] = SEED_PARTICION

ruta_particiones = PARTICIONES_DIR / "particiones_principal_60_20_20.parquet"
metadata_principal.to_parquet(ruta_particiones, index=False)